# Reproducing the Nonomura et al. (2019) EKFDMD example

This notebook reproduces the small-degree-of-freedom, **no system noise** numerical experiment from Section 4.1 of:

T. Nonomura, H. Shibata, and R. Takaki, *Extended-Kalman-filter-based dynamic mode decomposition for simultaneous system identification and denoising*, PLoS ONE 14(2), e0209836 (2019).  
https://doi.org/10.1371/journal.pone.0209836

The paper uses:

- a 6-dimensional linear continuous-time system;
- continuous eigenvalues `±2πi`, `±5πi`, and `-0.3 ± 11πi`;
- `Δt = 0.01`;
- random QR embedding from 6 to 16 dimensions;
- 500 snapshots;
- observation noise only;
- EKF initial covariance `P₀ = 10³ I`;
- `Q = 0`;
- `R = σ_w² I`;
- initial EKFDMD operator `A₀ = I`.

The paper averages many random realizations. This quickstart fixes one random seed so that the notebook is exactly reproducible. We use `σ_w² = 0.01`, one of the four noise levels reported in the paper.

Run this notebook from the repository root after installing the package in editable mode and installing Matplotlib:

```sh
python -m pip install -e . matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

from ekfdmd import EKFDMD

## 1. Construct the six-dimensional reference system

The paper defines a block-diagonal continuous-time system

\[
\dot f = B f,
\]

with three conjugate eigenvalue pairs. The exact discrete-time operator is

\[
A_6 = \exp(B\Delta t).
\]

In [ ]:
dt = 0.01
n_latent = 6
n_observed = 16
n_snapshots = 500
sigma_w2 = 0.01
seed = 7

omega = np.array([
    2j * np.pi,
    5j * np.pi,
    -0.3 + 11j * np.pi,
])

B = np.zeros((n_latent, n_latent))
for j, w in enumerate(omega):
    a, b = w.real, abs(w.imag)
    B[2*j:2*j+2, 2*j:2*j+2] = [[a, b], [-b, a]]

A6 = expm(B * dt)
true_eigs = np.linalg.eigvals(A6)

np.sort_complex(true_eigs)

The target discrete eigenvalues are therefore

\[
e^{\pm 2\pi i\Delta t}, \quad
e^{\pm 5\pi i\Delta t}, \quad
e^{(-0.3\pm 11\pi i)\Delta t}.
\]

In [ ]:
expected_eigs = np.array([
    np.exp( 2j * np.pi * dt),
    np.exp(-2j * np.pi * dt),
    np.exp( 5j * np.pi * dt),
    np.exp(-5j * np.pi * dt),
    np.exp((-0.3 + 11j * np.pi) * dt),
    np.exp((-0.3 - 11j * np.pi) * dt),
])

np.testing.assert_allclose(
    np.sort_complex(true_eigs),
    np.sort_complex(expected_eigs),
    atol=1e-12,
)

## 2. Generate the snapshots used in Section 4.1

The paper draws the initial latent state componentwise from a normal distribution centered at 1, embeds the six-dimensional trajectory into 16 dimensions using the `Q` factor of a random QR decomposition, and then adds white observation noise.

The exact random seed is not prescribed in the paper, so this notebook fixes one seed.

In [ ]:
rng = np.random.default_rng(seed)

f = np.empty((n_latent, n_snapshots))
f[:, 0] = rng.normal(loc=1.0, scale=np.sqrt(0.1), size=n_latent)

for k in range(1, n_snapshots):
    f[:, k] = A6 @ f[:, k - 1]

Q_qr, _ = np.linalg.qr(rng.normal(size=(n_observed, n_latent)), mode="reduced")
X_true = Q_qr @ f
Y = X_true + rng.normal(scale=np.sqrt(sigma_w2), size=X_true.shape)

X_true.shape, Y.shape

## 3. Fit EKFDMD with the paper's Kalman-filter settings

For this experiment the paper specifies

\[
P_0 = 10^3 I,\qquad Q=0,\qquad R=\sigma_w^2 I,
\]

and initializes the operator with the identity matrix. These choices map directly to `EKFDMD` as follows.

In [ ]:
model = EKFDMD(
    state_noise=0.0,
    operator_noise=0.0,
    measurement_noise=sigma_w2,
    initial_covariance=1e3,
    initialize_with_dmd=False,
).fit(Y)

model.operator.shape

## 4. Compare the estimated and exact eigenvalues

The 16-dimensional embedded system contains six dynamically active eigenvalues. The remaining directions are outside the generated six-dimensional subspace, so the comparison below matches each exact eigenvalue with the nearest eigenvalue of the final EKFDMD operator.

In [ ]:
estimated_eigs = np.linalg.eigvals(model.operator)

matched = []
remaining = list(estimated_eigs)
for target in true_eigs:
    index = int(np.argmin(np.abs(np.asarray(remaining) - target)))
    matched.append(remaining.pop(index))
matched = np.asarray(matched)

eigenvalue_error = np.abs(matched - true_eigs)

for target, estimate, error in zip(true_eigs, matched, eigenvalue_error):
    print(f"true={target: .6f}  estimated={estimate: .6f}  |error|={error:.3e}")

print(f"mean absolute eigenvalue error: {eigenvalue_error.mean():.3e}")

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(np.cos(theta), np.sin(theta), "--", linewidth=1, label="unit circle")
ax.scatter(estimated_eigs.real, estimated_eigs.imag, s=24, alpha=0.45, label="EKFDMD: all 16")
ax.scatter(true_eigs.real, true_eigs.imag, marker="x", s=80, linewidths=2, label="true active eigenvalues")
ax.set_aspect("equal")
ax.set_xlabel("Real")
ax.set_ylabel("Imaginary")
ax.set_title(rf"Nonomura et al. (2019), $\sigma_w^2={sigma_w2}$")
ax.grid(True)
ax.legend()
plt.show()

## 5. Simultaneous denoising

A distinguishing feature of EKFDMD in the paper is that state estimation and operator identification are performed simultaneously. `filtered_data` contains the EKF posterior state at each snapshot.

The plot below shows one observed coordinate. The filtered trajectory should follow the noise-free reference more closely than the raw observation.

In [ ]:
coordinate = 0
time = np.arange(n_snapshots) * dt

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time, Y[coordinate], linewidth=0.8, alpha=0.45, label="observation")
ax.plot(time, X_true[coordinate], linewidth=1.5, label="true")
ax.plot(time, model.filtered_data[coordinate], linewidth=1.2, label="EKFDMD filtered")
ax.set_xlabel("time")
ax.set_ylabel(r"$x_1$")
ax.grid(True)
ax.legend()
plt.show()

raw_rmse = np.sqrt(np.mean((Y - X_true) ** 2))
filtered_rmse = np.sqrt(np.mean((model.filtered_data - X_true) ** 2))

print(f"observation RMSE: {raw_rmse:.4f}")
print(f"filtered RMSE:    {filtered_rmse:.4f}")

## 6. Try the other noise levels from the paper

Section 4.1 reports `σ_w² ∈ {1e-4, 1e-3, 1e-2, 1e-1}`. To reproduce another case, change only `sigma_w2` near the top of the notebook and rerun all cells.

The paper evaluates 100 random realizations at each noise level. This quickstart intentionally keeps a single deterministic realization so that it remains small and fast enough to serve as an introductory example. A Monte Carlo reproduction belongs in a separate validation/benchmark script if it is needed later.